# 03 — Unified Evaluation (11 models)

Load semua model weights secara sequential, jalankan evaluasi standar.

**Output:** `artifacts/logs/evaluation/results_master.csv`

> ⚠️ Setiap model di-load lalu di-unload sebelum model berikutnya untuk menghindari OOM.

In [ ]:
# ── Cell 1: Install ──────────────────────────────────────────────────────────
!pip install ultralytics openmim pyyaml tqdm --quiet
!mim install mmengine mmcv mmpose --quiet

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Environment & Path Setup — kompatibel Kaggle DAN Colab (auto-detect)
# ══════════════════════════════════════════════════════════════════════
import os, sys

def _detect_env():
    if os.path.exists('/kaggle/working'):
        return 'kaggle'
    if os.path.exists('/content'):
        return 'colab'
    return 'local'

ENV = _detect_env()
print(f'Environment terdeteksi: {ENV}')

# ⚠️ GANTI dengan URL repo GitHub kamu sendiri (isi SEKALI saja per notebook)
GITHUB_REPO = 'https://github.com/atangorp/soccer-homography-benchmark.git'

def _link(src, dst):
    """Symlink src -> dst kalau src ada dan dst belum ada. Aman dipanggil berkali-kali."""
    if os.path.exists(src) and not os.path.exists(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        os.symlink(src, dst)

if ENV == 'kaggle':
    PROJECT_ROOT = '/kaggle/working/soccer-homography-benchmark'
    if not os.path.exists(PROJECT_ROOT):
        os.system(f'git clone -q {GITHUB_REPO} "{PROJECT_ROOT}"')
    sys.path.insert(0, PROJECT_ROOT)

    # WAJIB: attach dataset "soccer-field-raw" via "+ Add Input" di sidebar kanan
    # (ganti 'soccer-field-raw' kalau slug dataset kamu berbeda)
    _link('/kaggle/input/soccer-field-raw',
          f'{PROJECT_ROOT}/data/raw/soccer-field-localization.v9i.yolov8')
    # Weights hasil training — attach masing-masing notebook output via "+ Add Input"
    _link('/kaggle/input/02a-train-yolo11/soccer-homography-benchmark/artifacts/weights/yolo11',
          f'{PROJECT_ROOT}/artifacts/weights/yolo11')
    _link('/kaggle/input/02b-train-hrnet/soccer-homography-benchmark/artifacts/weights/hrnet',
          f'{PROJECT_ROOT}/artifacts/weights/hrnet')
    _link('/kaggle/input/02c-train-vitpose/soccer-homography-benchmark/artifacts/weights/vitpose',
          f'{PROJECT_ROOT}/artifacts/weights/vitpose')
    _link('/kaggle/input/02d-train-detr/soccer-homography-benchmark/artifacts/weights/detr',
          f'{PROJECT_ROOT}/artifacts/weights/detr')
    # Gambar test set (dibutuhkan evaluasi visual + GT label)
    for _split in ['train', 'val', 'test']:
        _link(f'{PROJECT_ROOT}/data/raw/soccer-field-localization.v9i.yolov8/{_split}/images',
              f'{PROJECT_ROOT}/data/converted/images/{_split}')

elif ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Colab Notebooks/soccer-homography-benchmark'
    sys.path.insert(0, PROJECT_ROOT)

else:
    PROJECT_ROOT = os.getcwd()
    sys.path.insert(0, PROJECT_ROOT)

import yaml, pandas as pd, torch
with open(f'{PROJECT_ROOT}/configs/eval_global.yaml') as f:
    CFG = yaml.safe_load(f)

TEST_IMAGES = f'{PROJECT_ROOT}/data/converted/images/test'
TEST_LABELS = f'{PROJECT_ROOT}/data/raw/soccer-field-localization.v9i.yolov8/test/labels'
OUTPUT_DIR  = f'{PROJECT_ROOT}/artifacts/logs/evaluation'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Config loaded: conf={CFG["conf_threshold"]}, ransac={CFG["ransac_threshold"]}')
print(f'PROJECT_ROOT : {PROJECT_ROOT}')


In [ ]:
# ── Cell 3: Definisi semua 11 model ──────────────────────────────────────────
# Edit path sesuai lokasi weights kalian.
W = f'{PROJECT_ROOT}/artifacts/weights'

MODEL_REGISTRY = [
    # ── YOLO11 ──
    {'family': 'yolo11', 'variant': 'small',  'type': 'yolo11',
     'weights': f'{W}/yolo11/small/run/weights/best.pt'},
    {'family': 'yolo11', 'variant': 'medium', 'type': 'yolo11',
     'weights': f'{W}/yolo11/medium/run/weights/best.pt'},
    {'family': 'yolo11', 'variant': 'xlarge', 'type': 'yolo11',
     'weights': f'{W}/yolo11/xlarge/run/weights/best.pt'},
    # ── HRNet ──
    {'family': 'hrnet', 'variant': 'w18', 'type': 'hrnet',
     'weights': f'{W}/hrnet/w18/best_coco_AP.pth',
     'config':  f'{W}/hrnet/w18/hrnet_w18_config.py'},
    {'family': 'hrnet', 'variant': 'w32', 'type': 'hrnet',
     'weights': f'{W}/hrnet/w32/best_coco_AP.pth',
     'config':  f'{W}/hrnet/w32/hrnet_w32_config.py'},
    {'family': 'hrnet', 'variant': 'w48', 'type': 'hrnet',
     'weights': f'{W}/hrnet/w48/best_coco_AP.pth',
     'config':  f'{W}/hrnet/w48/hrnet_w48_config.py'},
    # ── ViTPose ──
    {'family': 'vitpose', 'variant': 'small', 'type': 'vitpose',
     'weights': f'{W}/vitpose/small/best_coco_AP.pth',
     'config':  f'{W}/vitpose/small/vitpose_small_config.py'},
    {'family': 'vitpose', 'variant': 'base',  'type': 'vitpose',
     'weights': f'{W}/vitpose/base/best_coco_AP.pth',
     'config':  f'{W}/vitpose/base/vitpose_base_config.py'},
    {'family': 'vitpose', 'variant': 'large', 'type': 'vitpose',
     'weights': f'{W}/vitpose/large/best_coco_AP.pth',
     'config':  f'{W}/vitpose/large/vitpose_large_config.py'},
    # ── DETR ──
    {'family': 'detr', 'variant': 'r50',  'type': 'detr',
     'weights': f'{W}/detr/r50/checkpoint_best.pth'},
    {'family': 'detr', 'variant': 'r101', 'type': 'detr',
     'weights': f'{W}/detr/r101/checkpoint_best.pth'},
]

# Cek weights mana yang sudah tersedia
print(f'{'Model':<25} {'Weights Exist':>15}')
print('-'*42)
for m in MODEL_REGISTRY:
    exists = os.path.exists(m['weights'])
    print(f"{m['family']}-{m['variant']:<20} {'✅' if exists else '⏳ belum':>15}")

In [ ]:
# ── Cell 4: Factory function ─────────────────────────────────────────────────
from src.models.yolo11_wrapper  import YOLO11Wrapper
from src.models.hrnet_wrapper   import HRNetWrapper
from src.models.vitpose_wrapper import ViTPoseWrapper
from src.models.detr_wrapper    import DeformableDETRWrapper
from src.evaluation.benchmarks  import run_benchmark, aggregate_results, save_results

def build_model(spec):
    t = spec['type']
    w = spec['weights']
    v = spec['variant']
    if t == 'yolo11':
        return YOLO11Wrapper(w, conf_threshold=CFG['conf_threshold'], variant=v)
    elif t == 'hrnet':
        return HRNetWrapper(w, spec['config'], conf_threshold=CFG['conf_threshold'], variant=v)
    elif t == 'vitpose':
        return ViTPoseWrapper(w, spec['config'], conf_threshold=CFG['conf_threshold'], variant=v)
    elif t == 'detr':
        return DeformableDETRWrapper(w, conf_threshold=CFG['conf_threshold'], variant=v)
    raise ValueError(f'Unknown type: {t}')

In [ ]:
# ── Cell 5: Run evaluation loop ──────────────────────────────────────────────
import gc

all_results = []
failed      = []

for spec in MODEL_REGISTRY:
    model_name = f"{spec['family']}-{spec['variant']}"
    
    if not os.path.exists(spec['weights']):
        print(f'\n⏭️  Skip {model_name} — weights belum ada')
        continue
    
    print(f'\n▶️  {model_name}')
    try:
        model = build_model(spec)
        model.load_model()
        
        df = run_benchmark(
            model=model,
            test_images_dir=TEST_IMAGES,
            test_labels_dir=TEST_LABELS,
            config=CFG,
        )
        all_results.append(df)
        
        # ⚠️ CRITICAL: unload sebelum model berikutnya
        model.unload()
        del model
        gc.collect()
        torch.cuda.empty_cache()
        print(f'   🧹 Memory cleared')
        
    except Exception as e:
        print(f'   ❌ Error: {e}')
        failed.append({'model': model_name, 'error': str(e)})
        gc.collect()
        torch.cuda.empty_cache()

print(f'\n{'='*55}')
print(f'Selesai: {len(all_results)} model evaluated, {len(failed)} failed')

In [ ]:
# ── Cell 6: Aggregate & save ─────────────────────────────────────────────────
if all_results:
    per_image_df = pd.concat(all_results, ignore_index=True)
    summary_df   = aggregate_results(all_results)
    save_results(per_image_df, summary_df, OUTPUT_DIR)
    
    print('\n📊 Results summary:')
    display(summary_df[['rank','model_full_name','detection_rate','mpe_mean','mpe_std','fps','model_size_mb']].round(2))
else:
    print('⚠️  Tidak ada hasil — semua model mungkin belum selesai training')